In [1]:
import pandas as pd

## 1. Borough-level disability aggregation, MEMS7GR only, three-tier participation


In [ ]:
import numpy as np

def normalize_columns(df):
    rename_map = {}
    for col in df.columns:
        if col.lower() == "age16plus":
            rename_map[col] = "Age16plus"
    return df.rename(columns=rename_map)

def select_annual_weight(df, value_col, valid_codes):
    postal_mask = pd.to_numeric(df["mode"], errors="coerce").eq(2)
    postal_has_valid_data = df.loc[postal_mask, value_col].isin(valid_codes).any()
    return "wt_final" if postal_has_valid_data else "wt_final_online"

def weighted_three_tier(sub, value_col, weight_col):
    valid = sub.dropna(subset=[value_col, weight_col])
    n = len(valid)
    if n == 0:
        return np.nan, np.nan, np.nan, 0, 0.0
    w = valid[weight_col]
    weighted_n = w.sum()
    if weighted_n == 0:
        return np.nan, np.nan, np.nan, n, 0.0
    inactive = w[valid[value_col] == 0].sum() / weighted_n
    fairly_active = w[valid[value_col] == 1].sum() / weighted_n
    active = w[valid[value_col] == 2].sum() / weighted_n
    return inactive, fairly_active, active, n, weighted_n

disab3_labels = {1: "limiting_disability", 2: "non_limiting_disability", 3: "no_disability"}

## 2. Aggregation function for a single year


In [ ]:
def aggregate_borough_disability_year(path, year_number):
    year_df = pd.read_csv(path)
    year_df = normalize_columns(year_df)
    year_df = year_df[year_df["Age16plus"] == 1].copy()

    mems7gr_cols_y = [c for c in year_df.columns if c.startswith("MEMS7GR_") and c != "MEMS7GR_ALL"]
    activities_y = sorted(c.replace("MEMS7GR_", "") for c in mems7gr_cols_y)
    disty_cols_y = [f"disty{i}_POP" for i in range(1, 14)]
    records = []

    for act in activities_y:
        weight_col = select_annual_weight(year_df, f"MEMS7GR_{act}", [0, 1, 2])
        cols_needed = [f"MEMS7GR_{act}", weight_col, "LA_2023"]
        sub_full = year_df[cols_needed + disty_cols_y + ["Disab3"]].copy()
        sub_full = sub_full.rename(columns={f"MEMS7GR_{act}": "MEMS7GR"})

        for la_val, la_sub in sub_full.groupby("LA_2023"):
            for dv, dsub in la_sub[la_sub["Disab3"].isin([1, 2, 3])].groupby("Disab3"):
                r0, r1, r2, n, wn = weighted_three_tier(dsub, "MEMS7GR", weight_col)
                records.append([year_number, la_val, disab3_labels[dv], act, r0, r1, r2, n, wn])

            for dcol in disty_cols_y:
                dsub = la_sub[la_sub[dcol] == 1]
                r0, r1, r2, n, wn = weighted_three_tier(dsub, "MEMS7GR", weight_col)
                records.append([year_number, la_val, dcol.replace("_POP", ""), act, r0, r1, r2, n, wn])

    return pd.DataFrame(records, columns=["year", "LA_2023", "disability_group", "activity", "inactive_rate", "fairly_active_rate", "active_rate", "n", "weighted_n"])

## 3. Test on Year 7


In [4]:
path = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year7_125activities.csv"

year7_result = aggregate_borough_disability_year(path, 7)
print(year7_result.shape)
print(year7_result.head(20))


(64000, 9)
    year  LA_2023         disability_group       activity  inactive_rate  \
0      7        8      limiting_disability  ABSEILING_H03            1.0   
1      7        8  non_limiting_disability  ABSEILING_H03            1.0   
2      7        8            no_disability  ABSEILING_H03            1.0   
3      7        8                   disty1  ABSEILING_H03            1.0   
4      7        8                   disty2  ABSEILING_H03            1.0   
5      7        8                   disty3  ABSEILING_H03            1.0   
6      7        8                   disty4  ABSEILING_H03            1.0   
7      7        8                   disty5  ABSEILING_H03            1.0   
8      7        8                   disty6  ABSEILING_H03            1.0   
9      7        8                   disty7  ABSEILING_H03            1.0   
10     7        8                   disty8  ABSEILING_H03            1.0   
11     7        8                   disty9  ABSEILING_H03            1.0   
1

## 4. File paths for all eight years
Only the 125-activity version is used.


In [5]:
import os

root = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20"

year_files = {
    1: os.path.join(root, "Yifeng Mao", "data", "active_lives_1516_london_125.csv"),
    2: os.path.join(root, "Yifeng Mao", "data", "active_lives_1617_london_125.csv"),
    3: os.path.join(root, "Siyan Xin", "2017~2018", "2017_data_125_activities.csv"),
    4: os.path.join(root, "Siyan Xin", "2018~2019", "2018_data_125_activities.csv"),
    5: os.path.join(root, "Shuhan Zhao", "docs", "1920_london32_stable125.csv"),
    6: os.path.join(root, "Shuhan Zhao", "docs", "2021_london32_stable125.csv"),
    7: os.path.join(root, "Jingyi Hua", "data", "processed", "year7_125activities.csv"),
    8: os.path.join(root, "Jingyi Hua", "data", "processed", "year8_125activities.csv"),
}

for year_number, file_path in year_files.items():
    print(year_number, os.path.exists(file_path), file_path)


1 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Yifeng Mao\data\active_lives_1516_london_125.csv
2 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Yifeng Mao\data\active_lives_1617_london_125.csv
3 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Siyan Xin\2017~2018\2017_data_125_activities.csv
4 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Siyan Xin\2018~2019\2018_data_125_activities.csv
5 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\docs\1920_london32_stable125.csv
6 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\docs\2021_london32_stable125.csv
7 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year7_125activities.csv
8 True C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\year8_125activities.csv


## 5. Check required columns before running all eight years


In [6]:
required_base_cols = ['Age16plus', 'Disab3', 'mode', 'wt_final', 'wt_final_online', 'LA_2023'] + [f'disty{i}_POP' for i in range(1, 14)]

def check_columns(path, year_number):
    cols = pd.read_csv(path, nrows=0).columns
    cols_lower = [c.lower() for c in cols]
    missing_base = [c for c in required_base_cols if c not in cols and c.lower() not in cols_lower]
    has_mems7gr = any(c.startswith('MEMS7GR_') for c in cols)
    print(f"Year {year_number}: missing base columns = {missing_base}, has MEMS7GR = {has_mems7gr}")

for year_number, file_path in year_files.items():
    check_columns(file_path, year_number)


Year 1: missing base columns = [], has MEMS7GR = True
Year 2: missing base columns = [], has MEMS7GR = True
Year 3: missing base columns = [], has MEMS7GR = True
Year 4: missing base columns = [], has MEMS7GR = True
Year 5: missing base columns = [], has MEMS7GR = True
Year 6: missing base columns = [], has MEMS7GR = True
Year 7: missing base columns = [], has MEMS7GR = True
Year 8: missing base columns = [], has MEMS7GR = True


## 6. Run aggregation for all eight years and combine


In [7]:
all_years_results = []
problem_years = {}

for year_number in sorted(year_files.keys()):
    print('Processing year', year_number)
    try:
        year_result = aggregate_borough_disability_year(year_files[year_number], year_number)
        all_years_results.append(year_result)
    except KeyError as e:
        temp_df = pd.read_csv(year_files[year_number], nrows=5)
        print(f'Year {year_number} is missing column {e}')
        print('Columns containing la, disab, disty, wt, or age in this file:')
        print([col for col in temp_df.columns
               if 'la_' in col.lower() or 'disab' in col.lower() or 'disty' in col.lower()
               or 'wt' in col.lower() or 'age' in col.lower()])
        problem_years[year_number] = list(temp_df.columns)

if all_years_results:
    final_borough_disability_table = pd.concat(all_years_results, ignore_index=True)
    print(final_borough_disability_table.shape)

print('Years with problems:', list(problem_years.keys()))


Processing year 1
Processing year 2
Processing year 3
Processing year 4
Processing year 5
Processing year 6
Processing year 7
Processing year 8
(514048, 9)
Years with problems: []


## 7. Save the final table


In [8]:
output_dir = r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed"
os.makedirs(output_dir, exist_ok=True)

In [9]:
final_output_path = os.path.join(output_dir, "RQ3_borough_disability_all_years.csv")
final_borough_disability_table.to_csv(final_output_path, index=False)
print('Saved to', final_output_path)

Saved to C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\RQ3_borough_disability_all_years.csv


## 8. Borough-level disability aggregation, overall activity level (not activity-specific)

In [ ]:
def aggregate_borough_disability_overall_year(path, year_number):
    year_df = pd.read_csv(path)
    year_df = normalize_columns(year_df)
    year_df = year_df[year_df["Age16plus"] == 1].copy()

    disty_cols_y = [f"disty{i}_POP" for i in range(1, 14)]
    cols_needed = ["MEMS7GR_ALL", "wt_final", "LA_2023"]
    sub_full = year_df[cols_needed + disty_cols_y + ["Disab3"]].copy()
    sub_full = sub_full.rename(columns={"MEMS7GR_ALL": "MEMS7GR"})
    records = []

    for la_val, la_sub in sub_full.groupby("LA_2023"):
        for dv, dsub in la_sub[la_sub["Disab3"].isin([1, 2, 3])].groupby("Disab3"):
            r0, r1, r2, n, wn = weighted_three_tier(dsub, "MEMS7GR", "wt_final")
            records.append([year_number, la_val, disab3_labels[dv], r0, r1, r2, n, wn])

        for dcol in disty_cols_y:
            dsub = la_sub[la_sub[dcol] == 1]
            r0, r1, r2, n, wn = weighted_three_tier(dsub, "MEMS7GR", "wt_final")
            records.append([year_number, la_val, dcol.replace("_POP", ""), r0, r1, r2, n, wn])

    return pd.DataFrame(records, columns=["year", "LA_2023", "disability_group", "inactive_rate", "fairly_active_rate", "active_rate", "n", "weighted_n"])

## 9. Check that MEMS7GR_ALL exists in every year before running all eight years

In [11]:
def check_mems7gr_all(path, year_number):
    cols = pd.read_csv(path, nrows=0).columns
    print(f"Year {year_number}: has MEMS7GR_ALL = {'MEMS7GR_ALL' in cols}")

for year_number, file_path in year_files.items():
    check_mems7gr_all(file_path, year_number)

Year 1: has MEMS7GR_ALL = True
Year 2: has MEMS7GR_ALL = True
Year 3: has MEMS7GR_ALL = True
Year 4: has MEMS7GR_ALL = True
Year 5: has MEMS7GR_ALL = True
Year 6: has MEMS7GR_ALL = True
Year 7: has MEMS7GR_ALL = True
Year 8: has MEMS7GR_ALL = True


## 10. Run overall-level aggregation for all eight years and combine


In [12]:
import time

t0 = time.time()
year1_df = pd.read_csv(year_files[1])
print('read_csv time:', time.time() - t0)
print('shape:', year1_df.shape)

t0 = time.time()
year1_df = normalize_columns(year1_df)
year1_filtered = year1_df[(year1_df['Age16plus'] == 1)].copy()
print('filter time:', time.time() - t0)
print('filtered shape:', year1_filtered.shape)

t0 = time.time()
test_result = aggregate_borough_disability_overall_year(year_files[1], 1)
print('full function time:', time.time() - t0)
print('result shape:', test_result.shape)

read_csv time: 1.6583728790283203
shape: (19620, 533)
filter time: 0.11159634590148926
filtered shape: (19620, 533)
full function time: 3.2963883876800537
result shape: (512, 8)


In [13]:
all_years_overall_results = []
problem_years_overall = {}

for year_number in sorted(year_files.keys()):
    print('Processing year', year_number)
    try:
        year_overall_result = aggregate_borough_disability_overall_year(year_files[year_number], year_number)
        all_years_overall_results.append(year_overall_result)
    except KeyError as e:
        temp_df = pd.read_csv(year_files[year_number], nrows=5)
        print(f'Year {year_number} is missing column {e}')
        problem_years_overall[year_number] = list(temp_df.columns)

if all_years_overall_results:
    final_borough_disability_overall_table = pd.concat(all_years_overall_results, ignore_index=True)
    print(final_borough_disability_overall_table.shape)

print('Years with problems:', list(problem_years_overall.keys()))

Processing year 1
Processing year 2
Processing year 3
Processing year 4
Processing year 5
Processing year 6
Processing year 7
Processing year 8
(4096, 8)
Years with problems: []


## 11. Save the overall-level table

In [14]:
overall_output_path = os.path.join(output_dir, "RQ3_borough_disability_overall_all_years.csv")
final_borough_disability_overall_table.to_csv(overall_output_path, index=False)
print('Saved to', overall_output_path)

Saved to C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\RQ3_borough_disability_overall_all_years.csv


In [1]:
import os
import pandas as pd

output_dir = (
    r"C:\Users\Lenovo\Documents\GitHub"
    r"\London-Sport2-Group20\Jingyi Hua\data\processed"
)

source_path = os.path.join(
    output_dir,
    "RQ3_borough_disability_all_years.csv"
)

final_path = os.path.join(
    output_dir,
    "RQ3_borough_disability_MEMS7GR_all_years.csv"
)

WHITELIST_PATH = (
    r"C:\Users\Lenovo\Desktop\Dissertation\Data"
    r"\8_codebook\125_activities_composites_year1_to_year8.xlsx"
)

whitelist_df = pd.read_excel(
    WHITELIST_PATH,
    sheet_name="1_Stable composites",
    header=3
)

whitelist_125 = set(
    whitelist_df["DV suffix"]
    .dropna()
    .astype(str)
    .str.strip()
)

EXCLUDED_INCOMPLETE_ACTIVITIES = {"HULAHOOP_P27"}
common_124 = whitelist_125 - EXCLUDED_INCOMPLETE_ACTIVITIES

assert len(whitelist_125) == 125
assert "HULAHOOP_P27" in whitelist_125
assert len(common_124) == 124

df = pd.read_csv(source_path)

print("Rows before filtering:", len(df))
print("Activities before filtering:", df["activity"].nunique())

df = df[df["activity"].isin(common_124)].copy()

print("Rows after filtering:", len(df))
print("Activities after filtering:", df["activity"].nunique())

assert df["activity"].nunique() == 124
assert "HULAHOOP_P27" not in set(df["activity"].unique())
assert set(df["activity"].unique()) == common_124
assert len(df) == 507_904

df.to_csv(final_path, index=False)

print("Saved to:", final_path)

Rows before filtering: 514048
Activities before filtering: 130
Rows after filtering: 507904
Activities after filtering: 124
Saved to: C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\data\processed\RQ3_borough_disability_MEMS7GR_all_years.csv
